# Figure 1 | lncRNA quantification in GTEx and dGTEx

**a.** Number of genes annotated for each gene biotype in the GENCODE reference used by GTEx releases over time. Shaded area is the previous GTEx release (v10) to this release (v11). The number of annotated lncRNA genes increases by 17,123 from v10 to v11. This net increase differs from the 16,786 novel lncRNAs in (b) because lncRNA genes sharing any transcript or exon with the v10 annotation are conservatively classified as existing.

**b.** Number of genes in each biotype in GENCODE v47 used by GTEx v11, colored by the degree of change to the gene or transcript model between v10 and v11. Colored bars highlight the categories “novel” lncRNA gene and "existing" lncRNA gene.

**c.** Classification of novel and existing lncRNA genes based on their genomic context. The percentage of genes in each category for novel and existing lncRNAs is shown. Most novel lncRNAs are intergenic.

**d.** In four datasets: adult GTEx v11, developmental GTEx (dGTEx), Marmoset NHP-dGTEx and Macaque NHP-dGTEx. Left: the number of novel and existing lncRNAs detected as expressed vs background at FDR 5%. Grey bars indicate the total number of annotated genes in the category for adult GTEx and dGTEx, and the number of annotated genes with orthologs for Macaque and Marmoset. Right: The number of novel and existing lncRNAs expressed vs background at FDR 5% in combinations of the four datasets. 1,403 novel lncRNAs are detected across all four datasets, indicating conserved expression across human development and primate evolution.


## Setup


In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Figure size, plot orders, markers, and palettes
fig_width, fig_height, fig_dpi = 7.2, 5.0, 300
gene_type_order = ["lncRNA", "protein_coding", "pseudogene", "sRNA"]
panel_a_gene_types = ["sRNA", "pseudogene", "protein_coding", "lncRNA"]
change_order = ["removed_gene", "no_change", "modified_gene", "reassigned_gene", "new_gene"]
existing_change = {"no_change", "modified_gene", "reassigned_gene"}
lnc_split_order = ["novel_lncRNA", "existing_lncRNA"]
cat_order = [
    "intergenic", "bidirectional", "sense_overlapping", "antisense_overlapping",
    "sense_intronic", "antisense_intronic",
]
cat_pretty = [
    "intergenic", "bidirectional", "overlapping\n(sense)", "overlapping\n(antisense)",
    "intronic\n(sense)", "intronic\n(antisense)",
]
marker_map = {"sRNA": "^", "pseudogene": "v", "protein_coding": "s", "lncRNA": "o"}
cat_names = ["gtex_expr", "dgtex_expr", "macaque_expr", "marmoset_expr"]
dot_cols = ["in_gtex_quant", "in_dgtex_quant", "in_macaque_quant", "in_marmoset_quant"]

gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
    "sRNA": "sRNA",
    "lncRNA": "lncRNA",
}
gene_type_palette = {
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
    "sRNA": sns.color_palette("Paired", 12)[7],
}
change_palette = {
    "removed_gene": "#f3f3f3",
    "no_change": "#dddddd",
    "modified_gene": "#888888",
    "reassigned_gene": "#555555",
    "new_gene": "#222222",
}


## Paths

In [ ]:
gencode_tx_counts_path = Path("path/to/gencode_transcript_counts.txt")
gencode_class_path = Path("path/to/gencode_v39_v47_gene_classification.txt")
lnc_cat_path = Path("path/to/lncrna_category.txt")
gencode_biotype_tx_counts_path = Path("path/to/gencode_biotype_transcript_counts.txt")

gtex_null_path = Path("path/to/GTEx.null_quantifications.txt")
dgtex_null_path = Path("path/to/dGTEx.null_quantifications.txt")
macaque_null_path = Path("path/to/macaque.null_quantifications.txt")
marmoset_null_path = Path("path/to/marmoset.null_quantifications.txt")
orth_path = Path("path/to/dGTEx_nhpGTEx_liftedlncRNAs.tsv")

gtex_sample_meta_path = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
dgtex_sample_map_path = Path("path/to/dgtex_sample_tissue_detail.tsv")
macaque_sample_map_path = Path("path/to/macaque_sample_tissue_detail.tsv")
marmoset_sample_map_path = Path("path/to/marmoset_sample_tissue_detail.tsv")

fig_dir = Path("figures")
fig_pdf = fig_dir / "figure_1.pdf"
fig_png = fig_dir / "figure_1.png"


## Load data


### Gencode annotations


In [ ]:
gene_version_counts = pd.read_csv(gencode_tx_counts_path, sep=",")
print(f"Gene version counts: {len(gene_version_counts):,} rows")


In [ ]:
gencode_genes = pd.read_table(gencode_class_path, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
# Methods five categories; map transcript-level change_level values into gene-level bins.
gencode_genes["change_level_plot"] = gencode_genes["change_level"].replace({
    "modified_transcript": "modified_gene",
    "new_transcript": "modified_gene",
})
print(f"Gencode genes: {len(gencode_genes):,}")


In [ ]:
lncrna_cats = pd.read_table(lnc_cat_path)
lncrna_cats = pd.merge(
    lncrna_cats,
    gencode_genes[["gene_id_stable", "gene_type_grouped"]],
    on="gene_id_stable", how="left",
)
lncrna_cats["genomic_context_plot"] = lncrna_cats["genomic_context"]
print(f"lncRNA categories: {len(lncrna_cats):,}")


In [ ]:
gene_biotype_tx_counts = pd.read_csv(gencode_biotype_tx_counts_path)
print(f"Biotype transcript counts: {len(gene_biotype_tx_counts):,} rows")


### Pooled intergenic-null tables

In [ ]:
gtex_null = pd.read_table(gtex_null_path)
print(f"GTEx null genes: {len(gtex_null):,}")


In [ ]:
dgtex_null = pd.read_csv(dgtex_null_path)
print(f"dGTEx null genes: {len(dgtex_null):,}")


In [ ]:
macaque_null = pd.read_csv(macaque_null_path)
print(f"Macaque null genes: {len(macaque_null):,}")


In [ ]:
marmoset_null = pd.read_csv(marmoset_null_path)
print(f"Marmoset null genes: {len(marmoset_null):,}")


In [ ]:
orth = pd.read_csv(orth_path, sep="\t")
for col in ["gene_id.macaque", "gene_id.marmoset"]:
    orth[col] = orth[col].replace(["NA", "-"], None)
print(f"Ortholog table: {len(orth):,}")


### Sample metadata


In [ ]:
gtex_sample_meta = pd.read_csv(gtex_sample_meta_path, sep="\t", low_memory=False)
n_gtex = gtex_sample_meta.loc[gtex_sample_meta["SMAFRZE"] == "RNASEQ", "SAMPID"].nunique()
print(f"adult GTEx v11 samples: {n_gtex:,}")


In [ ]:
dgtex_sample_map = pd.read_table(dgtex_sample_map_path)
n_dgtex = dgtex_sample_map["sample_id"].nunique()
print(f"dGTEx samples: {n_dgtex:,}")


In [ ]:
macaque_sample_map = pd.read_table(macaque_sample_map_path)
n_macaque = macaque_sample_map["sample_id"].nunique()
print(f"macaque samples: {n_macaque:,}")


In [ ]:
marmoset_sample_map = pd.read_table(marmoset_sample_map_path)
n_marmoset = marmoset_sample_map["sample_id"].nunique()
print(f"marmoset samples: {n_marmoset:,}")


## Run analysis


### Cross-species expression and quantified flags

In [ ]:
# Expressed = expressed_* flags from lncrna_category.txt; quantified = present in pooled null tables.
filtered = lncrna_cats.merge(
    gencode_genes[["gene_id_stable", "in_v47_full"]],
    on="gene_id_stable",
    how="left",
)
filtered = filtered.loc[filtered["in_v47_full"].fillna(False)].copy()

orth_by = orth.drop_duplicates("gene_id.human").set_index("gene_id.human")
mac_id = filtered["gene_id"].map(orth_by["gene_id.macaque"])
mar_id = filtered["gene_id"].map(orth_by["gene_id.marmoset"])

filtered["gtex_expr"] = filtered["expressed_gtex"].astype(bool)
filtered["dgtex_expr"] = filtered["expressed_dgtex"].astype(bool)
filtered["macaque_expr"] = filtered["expressed_macaque"].astype(bool)
filtered["marmoset_expr"] = filtered["expressed_marmoset"].astype(bool)

filtered["in_gtex_quant"] = filtered["gene_id"].isin(gtex_null["gene_id"])
filtered["in_dgtex_quant"] = filtered["gene_id"].isin(dgtex_null["Gene"])
filtered["in_macaque_quant"] = mac_id.isin(macaque_null["Gene"])
filtered["in_marmoset_quant"] = mar_id.isin(marmoset_null["Gene"])

filtered["detected_any"] = filtered[["gtex_expr", "dgtex_expr", "macaque_expr", "marmoset_expr"]].any(axis=1)
filtered["detected_all"] = filtered[["gtex_expr", "dgtex_expr", "macaque_expr", "marmoset_expr"]].all(axis=1)
print(
    f"lncRNAs in v47: {len(filtered):,}; "
    f"gtex={int(filtered['gtex_expr'].sum()):,}; "
    f"dgtex={int(filtered['dgtex_expr'].sum()):,}; "
    f"macaque={int(filtered['macaque_expr'].sum()):,}; "
    f"marmoset={int(filtered['marmoset_expr'].sum()):,}; "
    f"all-4={int(filtered['detected_all'].sum()):,}"
)


### UpSet intersection tables


In [ ]:
dataset_labels = [
    f"GTEx (n={n_gtex:,})", f"dGTEx (n={n_dgtex:,})",
    f"Macaque (n={n_macaque:,})", f"Marmoset (n={n_marmoset:,})",
]

combos = list(product([False, True], repeat=len(cat_names)))
upset_records = []
for combo in combos:
    combo_mask = np.ones(len(filtered), dtype=bool)
    for i, col in enumerate(cat_names):
        combo_mask &= filtered[col] == combo[i]
    for split_label in lnc_split_order:
        count = (combo_mask & (filtered["gene_type_grouped"] == split_label)).sum()
        if count > 0:
            upset_records.append({"combo": combo, "group": split_label, "count": int(count)})

upset_df = pd.DataFrame(upset_records)
combo_df = pd.DataFrame(upset_df["combo"].tolist(), columns=cat_names)
upset_df = pd.concat([combo_df, upset_df[["group", "count"]]], axis=1)
intersection_sums = (
    upset_df.groupby(cat_names, as_index=False)["count"].sum().sort_values("count", ascending=False).reset_index(drop=True)
)
combos_sorted = [tuple(row[cat_names].values) for _, row in intersection_sums.iterrows()]

intersection_stack = []
for split_label in lnc_split_order:
    split_counts = []
    for combo in combos_sorted:
        row = upset_df[((upset_df[cat_names] == combo).all(axis=1)) & (upset_df["group"] == split_label)]
        split_counts.append(row["count"].iloc[0] if len(row) else 0)
    intersection_stack.append(split_counts)
intersection_stack = np.array(intersection_stack)
x = np.arange(len(combos_sorted))

set_sizes = np.zeros((len(lnc_split_order), len(cat_names)))
for i, split_label in enumerate(lnc_split_order):
    split_mask = filtered["gene_type_grouped"] == split_label
    for j, col in enumerate(cat_names):
        set_sizes[i, j] = filtered.loc[split_mask, col].sum()

dot_counts = np.zeros((len(lnc_split_order), len(cat_names)))
for i, split_label in enumerate(lnc_split_order):
    split_mask = filtered["gene_type_grouped"] == split_label
    for j, col in enumerate(dot_cols):
        dot_counts[i, j] = filtered.loc[split_mask, col].sum()

stacked_histplot_df = pd.DataFrame([
    {"combo_index": i_combo, "group": split_label}
    for i_combo in range(len(combos_sorted))
    for i, split_label in enumerate(lnc_split_order)
    for _ in range(int(intersection_stack[i, i_combo]))
])
max_intersection = intersection_stack.sum(axis=0).max()
dot_spacing = max(max_intersection * 0.1, 1)
upset_y_positions = -np.arange(len(cat_names)) * dot_spacing
print(f"UpSet intersections: {len(combos_sorted)}; max bar height: {max_intersection:,}")


## Make figure


In [ ]:
lncrna_cats["genomic_context_plot"] = pd.Categorical(
    lncrna_cats["genomic_context_plot"].astype(str), categories=cat_order, ordered=True,
)

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs_main = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.25], hspace=0.4)

gs_top = gs_main[0].subgridspec(1, 3, width_ratios=[.5, .3, .2], wspace=0.75)
ax_left = fig.add_subplot(gs_top[0, 0])
ax_mid = fig.add_subplot(gs_top[0, 1])
ax_right = fig.add_subplot(gs_top[0, 2])

gs_upset = gs_main[1].subgridspec(
    2, 2, height_ratios=[.66, .33], width_ratios=[.2, .8], hspace=0.1, wspace=0.35,
)
ax_blank = fig.add_subplot(gs_upset[0, 0])
ax_bar = fig.add_subplot(gs_upset[0, 1])
ax_set = fig.add_subplot(gs_upset[1, 0])
ax_dot = fig.add_subplot(gs_upset[1, 1], sharex=ax_bar)

# Panel a: annotated gene counts over GTEx releases
lines, labels = [], []
for gt in panel_a_gene_types:
    sns.lineplot(
        x=gene_version_counts["gtex_year"], y=gene_version_counts[gt],
        marker=marker_map[gt], linewidth=0.8, markersize=6,
        color=gene_type_palette[gt], markerfacecolor=gene_type_palette[gt],
        markeredgecolor="k", markeredgewidth=0.35, ax=ax_left, legend=False,
    )
    lines.append(plt.Line2D(
        [0], [0], marker=marker_map[gt], color="w",
        markerfacecolor=gene_type_palette[gt], markeredgecolor="k",
        markeredgewidth=0.35, markersize=8, linewidth=.5,
    ))
    labels.append(gene_type_pretty[gt])
ax_left.set_xlabel("GTEx release year")
ax_left.set_ylabel("Annotated genes")
ax_left.set_xticks(np.arange(2012, 2025, 3))
ax_left.spines[["top", "right"]].set_visible(False)
ax_left.axvspan(2024, 2025, color="grey", alpha=0.2, zorder=-1, linewidth=0)
ax_left.set_yticks([0, 10000, 20000, 30000, 40000])
ax_left.get_yaxis().set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k" if x >= 10000 else f"{int(x):,}")
)
ax_left.text(2024.5, ax_left.get_ylim()[1] * 0.94, r"$v_{10} \rightarrow v_{11}$",
             ha="center", va="bottom", color="grey", fontsize=6)
ax_left.legend(lines, labels, loc="upper left", bbox_to_anchor=(0.02, 1),
               title="Biotype", frameon=False, markerscale=0.6)

# Panel b: v10→v11 change summary (collapse novel/existing → lncRNA for this plot only)
sns.histplot(
    data=gencode_genes.assign(
        gene_type_grouped=pd.Categorical(
            gencode_genes["gene_type_grouped"].replace(
                {"novel_lncRNA": "lncRNA", "existing_lncRNA": "lncRNA"}
            ),
            categories=gene_type_order,
            ordered=True,
        )
    ),
    y="gene_type_grouped", hue="change_level_plot", multiple="stack",
    palette=change_palette, hue_order=list(reversed(change_order)), ax=ax_mid,
    shrink=0.8, edgecolor="k", zorder=10, alpha=1, linewidth=1,
)
ax_mid.set_ylabel("")
ax_mid.set_xlabel("Genes")
ax_mid.spines[["top", "right"]].set_visible(False)
ax_mid.figure.canvas.draw()
for ytick in ax_mid.get_yticks():
    gt = gene_type_order[int(ytick)]
    ax_mid.plot(
        [-0.1], [ytick], marker=marker_map[gt], markersize=6, color="k",
        markerfacecolor=gene_type_palette[gt], markeredgecolor="none", markeredgewidth=1,
        clip_on=False, transform=ax_mid.get_yaxis_transform(which="grid"), zorder=10,
    )
ax_mid.set_yticklabels([gene_type_pretty[gt] for gt in gene_type_order])
ax_mid.tick_params(axis="y", pad=13, length=0)
ax_mid.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k" if x >= 10000 else f"{int(x):,}"))
ax_mid.legend(
    handles=[mpatches.Patch(facecolor=change_palette[cl], edgecolor="k") for cl in change_order],
    labels=[cl.replace("_", " ") for cl in change_order],
    title=r"$v_{10} \rightarrow v_{11}$ change", loc="lower right", frameon=False, bbox_to_anchor=(1.1, 0),
)
lncrna_idx = gene_type_order.index("lncRNA")
y_lncrna = ax_mid.get_yticks()[lncrna_idx]
lnc_mask = gencode_genes["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"])
x_left_existing = (lnc_mask & (gencode_genes["change_level_plot"] == "removed_gene")).sum()
x_right_existing = x_left_existing + (lnc_mask & gencode_genes["change_level_plot"].isin(existing_change)).sum()
x_left_novel = x_right_existing
x_right_novel = lnc_mask.sum()
ax_mid.add_patch(mpatches.Rectangle(
    (x_left_existing, y_lncrna - 0.4), x_right_existing - x_left_existing, 0.8,
    linewidth=1.5, edgecolor=gene_type_palette["existing_lncRNA"], facecolor="none", zorder=15,
))
ax_mid.text(x_left_existing + (x_right_existing - x_left_existing) / 2, y_lncrna - 0.45, "existing",
            ha="center", va="bottom", color=gene_type_palette["existing_lncRNA"], zorder=20, fontsize=6)
ax_mid.add_patch(mpatches.Rectangle(
    (x_left_novel, y_lncrna - 0.4), x_right_novel - x_left_novel, 0.8,
    linewidth=1.5, edgecolor=gene_type_palette["novel_lncRNA"], facecolor="none", zorder=15,
))
ax_mid.text(x_left_novel + (x_right_novel - x_left_novel) / 2, y_lncrna - 0.45, "novel",
            ha="center", va="bottom", color=gene_type_palette["novel_lncRNA"], zorder=20, fontsize=6)

# Panel c: lncRNA genomic categories
sns.histplot(
    data=lncrna_cats, y="genomic_context_plot", hue="gene_type_grouped",
    hue_order=lnc_split_order, multiple="dodge", palette=gene_type_palette, edgecolor="k", zorder=10,
    legend=True, stat="percent", shrink=0.8, alpha=1, common_norm=False, ax=ax_right, linewidth=1,
)
ax_right.set_xlabel("% lncRNA genes")
ax_right.set_ylabel("Genomic context")
ax_right.spines[["top", "right"]].set_visible(False)
ax_right.tick_params(axis="y", length=0)
ax_right.set_yticks(range(len(cat_order)))
ax_right.set_yticklabels(cat_pretty)
ax_right.set_xticks([0, 25, 50])
ax_right.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.move_legend(ax_right, "lower right", bbox_to_anchor=(1.1, 0), title="Annotation",
                labels=["novel", "existing"], frameon=False)

# Panel d: UpSet
sns.histplot(
    data=stacked_histplot_df, x="combo_index", hue="group", multiple="dodge", discrete=True,
    ax=ax_bar, palette=gene_type_palette, edgecolor="k", linewidth=1, shrink=0.8,
    legend=False, alpha=1, zorder=2,
)
ax_bar.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(abs(x) / 1000)}k" if abs(x) >= 1000 else f"{abs(int(x))}"))
ax_bar.set_ylabel("Expressed vs intergenic")
ax_bar.set_xlabel("")
ax_bar.set_xticks([])
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)

bar_width = 0.35
cat_indices = np.arange(len(cat_names))
offsets = np.linspace(-bar_width / 2, bar_width / 2, len(lnc_split_order))
max_bar = max(set_sizes.max(), dot_counts.max())
for i, split_label in enumerate(lnc_split_order):
    ax_set.barh(
        y=cat_indices + offsets[i], width=set_sizes[i],
        color=gene_type_palette[split_label], edgecolor="k", height=bar_width,
        label=gene_type_pretty[split_label], alpha=1, zorder=2, left=-set_sizes[i],
    )
ax_set.invert_yaxis()
ax_set.set_xlabel("Expressed vs intergenic")
ax_set.set_ylabel("")
ax_set.set_yticks(cat_indices)
ax_set.set_yticklabels([])
ax_set.tick_params(axis="y", left=False, length=0)
ax_set.spines["top"].set_visible(False)
ax_set.spines["left"].set_visible(False)
ax_set.spines["right"].set_visible(False)
ax_set.set_xlim(-max_bar * 1.05, 0)
ax_set.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(abs(x) / 1000)}k" if abs(x) >= 1000 else f"{abs(int(x))}"))
for i, split_label in enumerate(lnc_split_order):
    for j in range(len(cat_names)):
        if dot_counts[i, j] > 0:
            ax_set.barh(
                cat_indices[j] + offsets[i],
                width=-dot_counts[i, j], left=0,
                height=bar_width,
                color="lightgrey", edgecolor="none", zorder=1, alpha=0.6, clip_on=False
            )

ax_dot.set_ylim(upset_y_positions.min() - dot_spacing * 0.6, dot_spacing * 0.6)
ax_dot.set_yticks(upset_y_positions)
ax_dot.set_yticklabels(dataset_labels)
ax_dot.yaxis.tick_left()
ax_dot.tick_params(axis="y", left=False, pad=-5, length=5, width=1)
for spine in ["top", "left", "right", "bottom"]:
    ax_dot.spines[spine].set_visible(False)
for idx, combo in enumerate(combos_sorted):
    true_ys = []
    for j, present in enumerate(combo):
        y_dot = upset_y_positions[j]
        color = "black" if present else "lightgray"
        ax_dot.scatter(idx, y_dot, color=color, s=48, zorder=3)
        if present:
            true_ys.append(y_dot)
    if len(true_ys) >= 2:
        ax_dot.plot([idx] * len(true_ys), true_ys, color="black", linewidth=0.8, zorder=2)
ax_dot.set_xticks(x)
ax_dot.set_xticklabels([""] * len(x))
ax_dot.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
ax_bar.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
ax_blank.axis("off")

# Panel labels
_inv = fig.transFigure.inverted()
_x_ad = _inv.transform(ax_left.transAxes.transform((-0.15, 0)))[0]
for ax, label in [(ax_left, "a"), (ax_mid, "b"), (ax_right, "c"), (ax_blank, "d")]:
    _y = _inv.transform(ax.transAxes.transform((0, 1.1)))[1]
    _x = _x_ad if label in ("a", "d") else _inv.transform(ax.transAxes.transform((-0.15, 0)))[0]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


### Gencode lncRNA counts


In [ ]:
# Catalog sizes use Methods biotype grouping (lncRNA biotypes + TEC).
gene_counts = gene_version_counts.set_index("gtex_version")
tx_counts = gene_biotype_tx_counts.set_index("gtex_version")

lnc_v39_genes = int(gene_counts.loc["v10", "lncRNA"])
lnc_v47_genes = int(gene_counts.loc["v11", "lncRNA"])
lnc_v39_tx = int(tx_counts.loc["v10", "lncRNA"])
lnc_v47_tx = int(tx_counts.loc["v11", "lncRNA"])

lnc_in_v47 = gencode_genes[
    (gencode_genes["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"])) & gencode_genes["in_v47_full"]
]
novel_lncrnas = int((lnc_in_v47["gene_type_grouped"] == "novel_lncRNA").sum())
existing_lncrnas = int((lnc_in_v47["gene_type_grouped"] == "existing_lncRNA").sum())

print(f"lncRNA transcripts (GENCODE v47): {lnc_v47_tx:,}")
print(f"lncRNA genes (GENCODE v47): {lnc_v47_genes:,}")
print(f"lncRNA transcript increase (v47 vs v39): {100 * (lnc_v47_tx - lnc_v39_tx) / lnc_v39_tx:.0f}%")
print(f"lncRNA gene increase (v47 vs v39): {100 * (lnc_v47_genes - lnc_v39_genes) / lnc_v39_genes:.0f}%")
print(f"additional lncRNA genes annotated in v11 compared to v10: {lnc_v47_genes - lnc_v39_genes:,}")
print(f"novel lncRNA genes: {novel_lncrnas:,}")
print(f"existing lncRNA genes: {existing_lncrnas:,}")


### Cross-species detection


In [ ]:
for split_label in lnc_split_order:
    split_df = filtered[filtered["gene_type_grouped"] == split_label]
    total = len(split_df)
    detected = int(split_df["detected_any"].sum())
    pct = (detected / total * 100) if total > 0 else 0
    conserved = int(split_df["detected_all"].sum())
    print(
        f"{gene_type_pretty[split_label]} expressed in ≥1 dataset: "
        f"{detected:,} / {total:,} ({pct:.1f}%)"
    )
    print(f"{gene_type_pretty[split_label]} detected in all 4 datasets: {conserved:,}")


In [ ]:
split_df = lncrna_cats[lncrna_cats["gene_type_grouped"] == "novel_lncRNA"]
n_intergenic = int((split_df["genomic_context_plot"] == "intergenic").sum())
n_overlap = len(split_df) - n_intergenic
print(
    f"novel lncRNAs intergenic: "
    f"{n_intergenic:,} / {len(split_df):,} ({100 * n_intergenic / len(split_df):.0f}%)"
)
print(
    f"novel lncRNAs overlapping existing genes: "
    f"{n_overlap:,} / {len(split_df):,} ({100 * n_overlap / len(split_df):.0f}%)"
)


### Sample sizes


In [ ]:
print(f"adult GTEx v11 samples: {n_gtex:,}")
print(f"dGTEx samples: {n_dgtex:,}")
print(f"macaque samples: {n_macaque:,}")
print(f"marmoset samples: {n_marmoset:,}")
